# Starbucks vs McDonald's Snack Analysis

This notebook loads the Starbucks and McDonald's nutrition datasets, standardizes their schemas with PySpark, and produces descriptive comparisons. The analysis is descriptive rather than predictive.

In [ ]:
import os
import sys
from pathlib import Path

from pyspark.sql import SparkSession

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)

spark = (
    SparkSession.builder
    .appName("Starbucks_McDonalds_Snack_Analysis_Notebook")
    .master("local[*]")
    .config("spark.pyspark.python", sys.executable)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

In [ ]:
from src.data_loader import load_csv_if_present
from src.data_cleaning import standardize_datasets

starbucks_df = load_csv_if_present(
    spark, str(ROOT / "dataset/raw/starbucks_drinkMenu_expanded.csv"), "Starbucks"
)
mcdonalds_df = load_csv_if_present(
    spark, str(ROOT / "dataset/raw/menu.csv"), "McDonald's"
)

if starbucks_df is None or mcdonalds_df is None:
    raise FileNotFoundError("Required CSV datasets are missing from dataset/raw.")


In [ ]:
standardized_df = standardize_datasets(starbucks_df, mcdonalds_df).cache()
standardized_df.printSchema()
standardized_df.show(5, truncate=False)

In [ ]:
from src.analysis import (
    compare_restaurants,
    descriptive_statistics,
    relationship_summary,
    summarize_categories,
)

restaurant_summary = compare_restaurants(standardized_df)
category_summary = summarize_categories(standardized_df)
statistics = descriptive_statistics(standardized_df)
relationships = relationship_summary(standardized_df)

display(restaurant_summary)
display(category_summary.head(10))
display(statistics.head(12))
display(relationships)

## Interpretation

The comparison is descriptive. McDonald's items generally have higher average calories, fat, and sodium, while Starbucks drinks have slightly higher average sugars in this dataset. The source menus contain different product types and serving conventions, so these results are not causal and are not perfectly like-for-like comparisons.

In [ ]:
spark.stop()